# CT 2W Resistance Increase -- Import, Filtering & Rate Fits

Loads two CT run time-series exports, filters the `Catalyst 2W Resistance` signal to steady operation
(target temperature band, `Pressure > 19` bar, no N2), puts it on a time-on-stream axis, and fits several
empirical resistance-vs-time forms with moving-block bootstrap confidence intervals:

- `331C_R1` -- `CT-357P-331C-V4-TS2-R1` (950 C)
- `350C_R1` -- `CT-385P-350C-D2-TS1-R1` (1050 C)

Both exports are sparse/event-logged: each row typically has a timestamp plus a value in only one or two
of the other columns (sensors log asynchronously), so each numeric column is handled against its own
non-null timestamps rather than assuming a shared regular grid. Because the files are large, the
overview plots default to a **resampled** view (`RESAMPLE` in the configuration cell).

## Before you start: get the data

Download raw data files from Nominal (CT-357P-331C-V4-TS2-R1 and CT-385P-350C-D2-TS1 workbooks from the CT run template). Export the "Standard Plots" tab to a csv. This will result in a large file, if a lighter file is desired, export only the Catalyst 2W resistance, Catalyst Outlet Temp, H2, N2, CO2, and Pressure traces.

Then:

1. Put both CSVs in one folder anywhere on your computer.
2. In the **Configuration** cell below, point `DATA_DIR` at that folder (or set the `CT_DATA_DIR`
   environment variable), and check the names in `FILES` match your exported file names.
3. Run the notebook top to bottom. The rate-fit section takes several minutes (bootstrap refits).

Requires Python 3.9+ with `pandas`, `numpy`, `scipy`, and `matplotlib`
(`pip install pandas numpy scipy matplotlib`).

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import os
from pathlib import Path
from scipy.optimize import curve_fit
from scipy import stats

pd.set_option('display.max_columns', None)
plt.rcParams['figure.figsize'] = (12, 4)

## Configuration

The only settings most people need to change are in the cell below:

- **`DATA_DIR`** -- the folder holding your two exported CSVs. Either edit the default, or set the
  `CT_DATA_DIR` environment variable before launching Jupyter so the notebook works unedited on any
  machine (e.g. `set CT_DATA_DIR=C:\path\to\exports` on Windows, `export CT_DATA_DIR=~/ct_exports` on
  macOS/Linux).
- **`FILES`** -- the CSV file names inside `DATA_DIR`. Change these if your exports are named differently.
- **`OUTPUT_DIR`** -- where figures and `model_parameters.csv` are written (created if missing;
  `CT_OUTPUT_DIR` environment variable also works).

The cell checks that both files exist before anything slow runs. Analysis settings (burn-in, fit models,
fit window, bootstrap size) live in the configuration cell of the rate-fit section further down.

In [ ]:
# --- Where your data lives ---------------------------------------------------------------------
# Folder containing the exported CSVs. Default: the CT_DATA_DIR environment variable if set, otherwise
# the folder this notebook is run from. Replace with a literal path if you prefer, e.g.
# DATA_DIR = Path(r'C:\Users\you\Downloads\ct_exports')
DATA_DIR = Path(os.environ.get('CT_DATA_DIR', '.')).expanduser()

# Exported CSV file names (inside DATA_DIR), keyed by the run names used throughout the notebook.
FILES = {
    '331C_R1': 'CT-357P-331C-V4-TS2-R1.csv',
    '350C_R1': 'CT-385P-350C-D2-TS1-R1.csv',
}

# --- Where results go -------------------------------------------------------------------------
OUTPUT_DIR = Path(os.environ.get('CT_OUTPUT_DIR', 'outputs')).expanduser()

# --- Plotting -----------------------------------------------------------------------------------
# Resample rule applied per-column before the overview plots (mean within each bin).
# Set to None to plot raw (unaggregated) points instead -- much slower for these file sizes.
RESAMPLE = '30s'

# --- Checks (no need to edit) -----------------------------------------------------------------
FILES = {name: DATA_DIR / fname for name, fname in FILES.items()}
_missing = [p for p in FILES.values() if not p.is_file()]
if _missing:
    raise FileNotFoundError(
        'Could not find these data files:\n'
        + '\n'.join(f'  {p.resolve()}' for p in _missing)
        + '\nDownload/export them as described at the top of the notebook, then set DATA_DIR (or the '
          'CT_DATA_DIR environment variable) to the folder that contains them, or fix the names in FILES.'
    )
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f'Data:    {DATA_DIR.resolve()}')
for name, p in FILES.items():
    print(f'  {name}: {p.name}  ({p.stat().st_size / 1e6:,.0f} MB)')
print(f'Outputs: {OUTPUT_DIR.resolve()}')

## Load helper

Parses the `timestamp` column as UTC datetimes and coerces all other columns to numeric.

In [ ]:
def load_timeseries_csv(path):
    df = pd.read_csv(path, parse_dates=['timestamp'])
    df['timestamp'] = pd.to_datetime(df['timestamp'], utc=True)
    value_cols = [c for c in df.columns if c != 'timestamp']
    for c in value_cols:
        df[c] = pd.to_numeric(df[c], errors='coerce')
    df = df.set_index('timestamp').sort_index()
    return df, value_cols


## Load both files

In [ ]:
# Traces the analysis below needs. H2/CO2 are only used (from the 350C run) to report the flow rate.
REQUIRED_COLUMNS = ['Catalyst 2W Resistance', 'Catalyst Outlet Temp', 'Pressure', 'N2']
FLOW_COLUMNS = ['H2', 'CO2']

data = {}
for name, path in FILES.items():
    df, value_cols = load_timeseries_csv(path)
    missing = [c for c in REQUIRED_COLUMNS if c not in value_cols]
    if missing:
        raise KeyError(f"{path.name} is missing required trace(s) {missing}. Re-export it including "
                       f"{REQUIRED_COLUMNS + FLOW_COLUMNS} (column names must match exactly).")
    data[name] = {'df': df, 'value_cols': value_cols}
    print(f"{name}: {path.name}")
    print(f"  rows: {len(df):,}  |  span: {df.index.min()} -> {df.index.max()}")
    print(f"  columns: {value_cols}")
    print()

In [ ]:
data['331C_R1']['df'].head()


In [ ]:
data['350C_R1']['df'].head()


## Plot helper

For each value column: drop NaNs, optionally resample/mean-aggregate on the given rule, then plot as its own subplot (columns don't share a common sampling grid).

In [ ]:
def plot_all_columns(df, value_cols, title, resample=RESAMPLE):
    n = len(value_cols)
    fig, axes = plt.subplots(n, 1, figsize=(12, 3 * n), sharex=True)
    if n == 1:
        axes = [axes]
    for ax, col in zip(axes, value_cols):
        series = df[col].dropna()
        if resample is not None and not series.empty:
            series = series.resample(resample).mean().dropna()
        ax.plot(series.index, series.values, linewidth=0.8)
        ax.set_ylabel(col, fontsize=9)
        ax.grid(alpha=0.3)
    axes[-1].xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m-%d %H:%M'))
    fig.autofmt_xdate()
    fig.suptitle(title, y=1.0)
    fig.tight_layout()
    return fig


## Plot: CT-357P-331C-V4-TS2-R1

In [ ]:
fig1 = plot_all_columns(data['331C_R1']['df'], data['331C_R1']['value_cols'], 'CT-357P-331C-V4-TS2-R1')
plt.show()


## Plot: CT-385P-350C-D2-TS1-R1

In [ ]:
fig2 = plot_all_columns(data['350C_R1']['df'], data['350C_R1']['value_cols'], 'CT-385P-350C-D2-TS1-R1')
plt.show()


## Single-column lookup

Quick way to plot one column from one file without regenerating the full grid above.

In [ ]:
def plot_column(dataset_name, column, resample=RESAMPLE):
    df = data[dataset_name]['df']
    series = df[column].dropna()
    if resample is not None and not series.empty:
        series = series.resample(resample).mean().dropna()
    fig, ax = plt.subplots()
    ax.plot(series.index, series.values, linewidth=0.8)
    ax.set_ylabel(column)
    ax.set_title(f'{dataset_name}: {column}')
    ax.grid(alpha=0.3)
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m-%d %H:%M'))
    fig.autofmt_xdate()
    plt.show()

# Example:
# plot_column('331C_R1', 'Catalyst 2W Resistance')


---
## Temperature-filtered Catalyst Resistance (`merge_asof`, carry-forward)

**Note on matching strategy:** an earlier version of this matched `Catalyst Outlet Temp` to each
resistance reading with `direction='nearest'` and a 10-minute tolerance. That silently dropped any
resistance reading whose nearest temp/pressure log entry was more than 10 minutes away -- and because
this data is deadbanded (a sensor only re-logs when its value *changes*, so a steady reading can go
unlogged for hours), that discarded the majority of points during long steady holds. On the 331C file,
Pressure alone was unmatched (and dropped) for ~62% of resistance readings, which produced a spurious
~20-hour gap in the TOS plot around TOS = 90-110 hr even though the underlying process was continuously
in-band there -- it just wasn't re-logging Pressure during that stretch.

The fix: use `direction='backward'` ("carry the last logged value forward until a new one is logged"),
which is the correct semantics for log-on-change / deadbanded sensors, instead of nearest-in-time with a
tight tolerance. An optional `max_staleness` still guards against carrying forward a reading that's
implausibly old (e.g. a genuine sensor outage) -- left as `None` (unbounded) by default since the data
here can legitimately hold steady for many hours.

- **CT-357P-331C-V4-TS2-R1.csv**: TC1 (`Catalyst Outlet Temp`) = 950°C ± 10°C
- **CT-385P-350C-D2-TS1-R1.csv**: TC2 (`Catalyst Outlet Temp`) = 1050°C ± 10°C

In [ ]:
RESISTANCE_COL = 'Catalyst 2W Resistance'
TEMP_COL = 'Catalyst Outlet Temp'

# Per-file target temperature band. max_staleness caps how old a carried-forward
# temp/pressure reading may be before a resistance point is treated as unmatched;
# None = unbounded (appropriate here since Pressure/Temp can legitimately hold
# steady, unlogged, for hours under deadbanded logging).
FILTER_CONFIG = {
    '331C_R1': {'target_temp': 950,  'temp_tol': 10, 'max_staleness': None},  # 331C file, TC1
    '350C_R1': {'target_temp': 1050, 'temp_tol': 10, 'max_staleness': None},  # 350C file, TC2
}


In [ ]:
def filter_resistance_by_temp(df, resistance_col, temp_col, target_temp, temp_tol, max_staleness=None):
    """Carry-forward match resistance readings to the most recently logged temperature
    reading via merge_asof(direction='backward') -- the correct semantics for a
    deadbanded / log-on-change sensor, where the last logged value remains true until
    a new one is logged. Then keep only the resistance points whose matched
    temperature is within target_temp +/- temp_tol.

    max_staleness (optional Timedelta-like string, e.g. '6h') caps how old the carried-
    forward temp reading may be; None means unbounded.

    Returns (filtered, merged):
      filtered -- resistance + matched temp, restricted to the target temperature band
      merged   -- every resistance reading with its carried-forward temp match (pre-filter),
                  useful for sanity-checking match quality / coverage
    """
    res = df[resistance_col].dropna().rename(resistance_col).reset_index()
    temp = df[temp_col].dropna().rename(temp_col).reset_index()
    res = res.sort_values('timestamp')
    temp = temp.sort_values('timestamp')

    merged = pd.merge_asof(
        res, temp,
        on='timestamp',
        direction='backward',
        tolerance=pd.Timedelta(max_staleness) if max_staleness is not None else None,
    )

    unmatched = merged[temp_col].isna().sum()
    if unmatched:
        reason = f'older than {max_staleness}' if max_staleness is not None else 'before the first logged temp reading'
        print(f"  {unmatched:,} / {len(merged):,} resistance points had no usable temp reading ({reason}) -- dropped")
    merged = merged.dropna(subset=[temp_col])

    mask = (merged[temp_col] - target_temp).abs() <= temp_tol
    filtered = merged.loc[mask, ['timestamp', resistance_col, temp_col]].set_index('timestamp')
    return filtered, merged.set_index('timestamp')


In [ ]:
filtered_data = {}
for name, cfg in FILTER_CONFIG.items():
    df = data[name]['df']
    print(f"{name} (target {cfg['target_temp']}C +/- {cfg['temp_tol']}C):")
    filtered, merged = filter_resistance_by_temp(
        df, RESISTANCE_COL, TEMP_COL,
        cfg['target_temp'], cfg['temp_tol'], cfg['max_staleness'],
    )
    filtered_data[name] = {'filtered': filtered, 'merged': merged}
    kept_pct = 100 * len(filtered) / len(merged) if len(merged) else 0
    print(f"  kept {len(filtered):,} / {len(merged):,} resistance points ({kept_pct:.1f}%) in band")
    print()


### Filtered resistance vs. time

Gray = all nearest-matched resistance readings (pre-filter); colored = readings kept inside the target temperature band.

In [ ]:
fig, axes = plt.subplots(len(filtered_data), 1, figsize=(12, 4 * len(filtered_data)))
if len(filtered_data) == 1:
    axes = [axes]

for ax, (name, d) in zip(axes, filtered_data.items()):
    cfg = FILTER_CONFIG[name]
    merged, filtered = d['merged'], d['filtered']
    ax.plot(merged.index, merged[RESISTANCE_COL], '.', color='lightgray', markersize=2, label='all matched')
    ax.plot(filtered.index, filtered[RESISTANCE_COL], '.', color='C0', markersize=3,
            label=f"{cfg['target_temp']}C \u00b1 {cfg['temp_tol']}C")
    ax.set_ylabel(RESISTANCE_COL, fontsize=9)
    ax.set_title(name)
    ax.legend(loc='upper left', fontsize=8)
    ax.grid(alpha=0.3)
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m-%d %H:%M'))

fig.autofmt_xdate()
fig.tight_layout()
plt.show()


In [ ]:
filtered_data['331C_R1']['filtered'].head()


---
## Temperature + Pressure + N2 filtered Catalyst Resistance

Same carry-forward `merge_asof(direction='backward')` approach as above, with two added criteria:
`Pressure > 19` (bar) and `N2 == 0` (bar), matched the same way (last logged value carried forward,
within `max_staleness`) since both Pressure and N2 are deadbanded/logged on their own schedule.

N2 isn't just sensor noise here -- in both files it holds at exactly 0 for a majority of the run
(57% of the 331C record, 78% of the 350C record) with clear, sustained excursions to ~6-10, so it
reads as a real experimental condition (an N2-diluent phase vs. a no-N2 phase) rather than a
calibration blip. This filter keeps only the no-N2 phase.

Kept: `Catalyst Outlet Temp` within its file's target band **and** `Pressure > 19` **and** `N2 == 0`.

In [ ]:
PRESSURE_COL = 'Pressure'
PRESSURE_MIN = 19  # bar

N2_COL = 'N2'
N2_MAX = 1e-6  # N2 must be (effectively) exactly 0 -- both files log it as literal 0.0 in this phase


In [ ]:
def filter_resistance_by_temp_pressure_and_n2(df, resistance_col, temp_col, pressure_col, n2_col,
                                               target_temp, temp_tol, pressure_min, n2_max,
                                               max_staleness=None):
    """Like filter_resistance_by_temp, but adds carry-forward Pressure and N2 matches and requires
    pressure_col > pressure_min and n2_col <= n2_max, in addition to the temperature band.

    Returns (filtered, merged):
      filtered -- resistance + matched temp/pressure/N2, restricted to the target temperature band
                  AND pressure > pressure_min AND N2 <= n2_max
      merged   -- every resistance reading with its carried-forward temp/pressure/N2 matches
                  (pre-filter), for sanity-checking match coverage
    """
    res = df[resistance_col].dropna().rename(resistance_col).reset_index()
    temp = df[temp_col].dropna().rename(temp_col).reset_index()
    pres = df[pressure_col].dropna().rename(pressure_col).reset_index()
    n2 = df[n2_col].dropna().rename(n2_col).reset_index()
    res = res.sort_values('timestamp')
    temp = temp.sort_values('timestamp')
    pres = pres.sort_values('timestamp')
    n2 = n2.sort_values('timestamp')

    tol = pd.Timedelta(max_staleness) if max_staleness is not None else None
    merged = pd.merge_asof(res, temp, on='timestamp', direction='backward', tolerance=tol)
    merged = pd.merge_asof(merged, pres, on='timestamp', direction='backward', tolerance=tol)
    merged = pd.merge_asof(merged, n2, on='timestamp', direction='backward', tolerance=tol)

    unmatched_temp = merged[temp_col].isna().sum()
    unmatched_pressure = merged[pressure_col].isna().sum()
    unmatched_n2 = merged[n2_col].isna().sum()
    reason = f'older than {max_staleness}' if max_staleness is not None else 'before the first logged reading'
    if unmatched_temp:
        print(f"  {unmatched_temp:,} / {len(merged):,} resistance points had no usable temp reading ({reason}) -- dropped")
    if unmatched_pressure:
        print(f"  {unmatched_pressure:,} / {len(merged):,} resistance points had no usable pressure reading ({reason}) -- dropped")
    if unmatched_n2:
        print(f"  {unmatched_n2:,} / {len(merged):,} resistance points had no usable N2 reading ({reason}) -- dropped")
    merged = merged.dropna(subset=[temp_col, pressure_col, n2_col])

    mask = ((merged[temp_col] - target_temp).abs() <= temp_tol) & (merged[pressure_col] > pressure_min) \
           & (merged[n2_col] <= n2_max)
    filtered = merged.loc[mask, ['timestamp', resistance_col, temp_col, pressure_col, n2_col]].set_index('timestamp')
    return filtered, merged.set_index('timestamp')


In [ ]:
filtered_data_with_pressure = {}
for name, cfg in FILTER_CONFIG.items():
    df = data[name]['df']
    print(f"{name} (target {cfg['target_temp']}C +/- {cfg['temp_tol']}C, Pressure > {PRESSURE_MIN} bar, N2 <= {N2_MAX}):")
    filtered, merged = filter_resistance_by_temp_pressure_and_n2(
        df, RESISTANCE_COL, TEMP_COL, PRESSURE_COL, N2_COL,
        cfg['target_temp'], cfg['temp_tol'], PRESSURE_MIN, N2_MAX, cfg['max_staleness'],
    )
    filtered_data_with_pressure[name] = {'filtered': filtered, 'merged': merged}
    kept_pct = 100 * len(filtered) / len(merged) if len(merged) else 0
    print(f"  kept {len(filtered):,} / {len(merged):,} resistance points ({kept_pct:.1f}%) in band")
    print()


### Temp + pressure filtered resistance vs. time

Gray = all nearest-matched resistance readings (pre-filter); colored = readings kept inside the temperature band **and** above the pressure threshold.

In [ ]:
fig, axes = plt.subplots(len(filtered_data_with_pressure), 1, figsize=(12, 4 * len(filtered_data_with_pressure)))
if len(filtered_data_with_pressure) == 1:
    axes = [axes]

for ax, (name, d) in zip(axes, filtered_data_with_pressure.items()):
    cfg = FILTER_CONFIG[name]
    merged, filtered = d['merged'], d['filtered']
    ax.plot(merged.index, merged[RESISTANCE_COL], '.', color='lightgray', markersize=2, label='all matched')
    ax.plot(filtered.index, filtered[RESISTANCE_COL], '.', color='C1', markersize=3,
            label=f"{cfg['target_temp']}C \u00b1 {cfg['temp_tol']}C & P>{PRESSURE_MIN} bar")
    ax.set_ylabel(RESISTANCE_COL, fontsize=9)
    ax.set_title(name)
    ax.legend(loc='upper left', fontsize=8)
    ax.grid(alpha=0.3)
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m-%d %H:%M'))

fig.autofmt_xdate()
fig.tight_layout()
plt.show()


In [ ]:
filtered_data_with_pressure['331C_R1']['filtered'].head()


---
## TOS, 20 barg (hr)

"Time on stream" -- a cumulative-hours axis that only advances while the temperature + pressure filter above is satisfied, instead of wall-clock time. Elapsed time between two consecutive nearest-matched samples is only counted toward TOS when **both** samples are in-band; elapsed time spent outside the band (startup, upsets, between runs, etc.) is excluded rather than just skipped over on a wall-clock axis.

This reuses the full nearest-matched table from the temp+pressure filter step (`filtered_data_with_pressure[name]['merged']`), which already carries the matched `Catalyst Outlet Temp` and `Pressure` per resistance reading.

In [ ]:
TOS_COL = 'TOS, 20 barg (hr)'

def add_time_on_stream(merged, in_band):
    """Add a cumulative time-on-stream column (hours) to `merged`, where elapsed time
    only accrues over intervals where both endpoints satisfy `in_band`.

    merged   -- DataFrame indexed by timestamp (sorted), e.g. filtered_data_with_pressure[name]['merged']
    in_band  -- boolean Series aligned to merged.index, True where the filter criteria hold

    Returns a copy of `merged` with the TOS_COL column added.
    """
    merged = merged.sort_index().copy()
    in_band = in_band.reindex(merged.index)

    dt = merged.index.to_series().diff().fillna(pd.Timedelta(0))
    both_in_band = in_band & in_band.shift(1, fill_value=False)
    dt_counted = dt.where(both_in_band.to_numpy(), pd.Timedelta(0))

    merged[TOS_COL] = dt_counted.cumsum().dt.total_seconds() / 3600.0
    return merged


In [ ]:
tos_data = {}
for name, cfg in FILTER_CONFIG.items():
    merged = filtered_data_with_pressure[name]['merged']
    in_band = (
        ((merged[TEMP_COL] - cfg['target_temp']).abs() <= cfg['temp_tol'])
        & (merged[PRESSURE_COL] > PRESSURE_MIN)
        & (merged[N2_COL] <= N2_MAX)
    )

    with_tos = add_time_on_stream(merged, in_band)
    filtered_with_tos = with_tos.loc[in_band.reindex(with_tos.index).fillna(False)]

    tos_data[name] = {'full': with_tos, 'filtered': filtered_with_tos}
    total_tos = filtered_with_tos[TOS_COL].max() if len(filtered_with_tos) else 0
    print(f"{name}: {len(filtered_with_tos):,} in-band points  |  total {TOS_COL} = {total_tos:.2f} hr")


### Resistance vs. TOS (per file)

In [ ]:
fig, axes = plt.subplots(len(tos_data), 1, figsize=(12, 4 * len(tos_data)))
if len(tos_data) == 1:
    axes = [axes]

for ax, (name, d) in zip(axes, tos_data.items()):
    cfg = FILTER_CONFIG[name]
    filtered = d['filtered']
    ax.plot(filtered[TOS_COL], filtered[RESISTANCE_COL], '.', color='C2', markersize=3)
    ax.set_xlabel(TOS_COL)
    ax.set_ylabel(RESISTANCE_COL, fontsize=9)
    ax.set_title(f"{name} ({cfg['target_temp']}C \u00b1 {cfg['temp_tol']}C & P>{PRESSURE_MIN} bar)")
    ax.grid(alpha=0.3)

fig.tight_layout()
plt.show()


### Resistance vs. TOS -- both files overlaid

Both runs start at TOS = 0, so degradation trends are directly comparable regardless of when each run actually happened in wall-clock time.

In [ ]:
fig, ax = plt.subplots()
for (name, d), color in zip(tos_data.items(), ['C0', 'C1']):
    filtered = d['filtered']
    ax.plot(filtered[TOS_COL], filtered[RESISTANCE_COL], '.', markersize=3, color=color, label=name)

ax.set_xlabel(TOS_COL)
ax.set_ylabel(RESISTANCE_COL)
ax.set_title(f'{RESISTANCE_COL} vs {TOS_COL}')
ax.legend()
ax.grid(alpha=0.3)
plt.show()


In [ ]:
tos_data['331C_R1']['filtered'][[RESISTANCE_COL, TEMP_COL, PRESSURE_COL, TOS_COL]].head()


---
# 2W Resistance Increase Rate -- linear / exponential / quadratic / power-law / geometric fits (post burn-in)

Fits each run's temp + pressure filtered `Catalyst 2W Resistance` vs. `TOS, 20 barg (hr)` (built above in
`tos_data`) after a fixed 10 h burn-in, using simple empirical models (choose which via `FIT_MODELS`):

- **Linear:** `R(t) = R0 + k*t`  -- `k` in Ohm/hr
- **Exponential:** `R(t) = R0 * exp(b*t)`  -- `b` is the fractional growth per hour (`100*(e^b - 1)` = %/hr)
- **Quadratic:** `R(t) = R0 + k*t + c*t^2`  -- a "parabola" in time: `c > 0` means the rate accelerates.
  Its slope changes over the record (`k + 2ct`), so it's summarized as the **average slope over the fit
  window**, `k + c*T` (= `[R(T) - R(0)] / T`, `T` = window length). Note this is a polynomial, not the
  parabolic oxidation law `R ~ sqrt(t)` (that one decelerates, the opposite of what this data does).
- **Power law:** `R(t) = R0 + A*t^n` with the exponent `n` fitted, bounded to `POWER_N_BOUNDS`
  (default `n >= 1`, i.e. accelerating). Generalizes the quadratic (`n = 2`) and linear (`n = 1`) shapes.
  Summarized the same way, as the average slope over the window: `A*T^(n-1)`. Note that with `n > 1`
  its slope at `t = 0` is forced to zero.
- **Linear + power law:** `R(t) = R0 + k*t + A*t^n` (`n` in `POWER_N_BOUNDS`). The `k*t` term lets the
  curve start with a non-zero slope at the end of burn-in, as the quadratic can. Average slope over the
  window: `k + A*T^(n-1)`. Four parameters, so `k`, `A`, and `n` can trade off against each other --
  check the bootstrap CI on `n` before reading much into it.
- **Geometric (Auriga paralinear doc, single segment):** `R(t) = R0 / (1 - A*t^n)`, fitted as
  `R0 / (1 - (t/t_run)^n)` so the runaway time `t_run` (where `R -> infinity`) is a parameter, constrained
  past the end of the fit window. Reads resistance as loss of conducting cross-section. With `n < 1` it
  starts steep and then accelerates, using one exponent (`n` in `GEOM_N_BOUNDS`, free below 1). Average slope
  over the window: `[R(T) - R0] / T`. The doc's per-pressure-segment `kv*ramp` terms drop out here (one
  20 bar segment), and its physics is for the 4W signal -- on 2W this is a descriptive curve, and
  `t_run` is an extrapolation, not a prediction.

where `t` is hours since the end of burn-in.

**Common fit window:** the curved models' average slope depends on the window length `T`, and with an
accelerating trend a longer run's average is pulled up by its late, steep stretch. By default
(`FIT_END_HOURS = 'common'`) every run is fit only up to the shortest run's end, so all rates cover
the same TOS span and are directly comparable. Set it to `None` to fit each run's full record, or to a
number of TOS hours.

No oxidation mechanism (parabolic / paralinear) is assumed:
neither run has a usable 4W signal, and the 2W signal mixes bulk material change with contact
degradation, so these fits are descriptive rates only -- not kinetic parameters.

**Uncertainty:** moving-block residual bootstrap. Residuals from the base fit are resampled in contiguous
blocks (preserving their autocorrelation), added back onto the fitted curve, and the model is refit;
95% CIs are the 2.5 / 97.5 percentiles of the refit parameters.

In [ ]:
BURN_IN_HOURS = 10          # exclude early-hours transients
BIN_HOURS = 1.0             # hourly-bin the densely logged raw series before fitting

# End of the fit window (TOS hours), applied to every run:
#   'common' -> the shortest run's usable end (after excursion masking), so rates are comparable
#   None     -> each run's full record
#   number   -> a fixed TOS cutoff
FIT_END_HOURS = 'common'

# Which models to fit / plot -- any subset of 'linear', 'exponential', 'quadratic'.
FIT_MODELS = ['linear', 'exponential', 'quadratic', 'power', 'linear_power', 'geometric']
POWER_N_BOUNDS = (1.0, 5.0)  # allowed range for the power-law exponent n
GEOM_N_BOUNDS = (0.05, 5.0)  # allowed range for the geometric-form exponent n
GEOM_TRUN_MAX_MULT = 1000    # runaway time bounded to (T, 1000*T] hours after burn-in

# Excursion masking: a rolling-window |slope| far above the run's typical value flags a transient
# upset (e.g. the 331C run's ~302-315h shutdown/recovery), not steady degradation. Data from the
# start of a flagged excursion onward is dropped, since the post-upset resistance level isn't
# continuous with the pre-upset trend. Set EXCLUDE_EXCURSIONS = False to fit the full record.
EXCLUDE_EXCURSIONS = True
EXCURSION_WINDOW_HOURS = 15
EXCURSION_STEP_HOURS = 2
EXCURSION_SLOPE_MULT = 5.0

BOOT_BLOCK_HOURS = 30       # moving-block bootstrap block length (hours)
N_BOOT = 1000
RNG = np.random.default_rng(42)

# Half-width (hours) of the local window around t = burn-in used to estimate the measured baseline
# resistance for the %/hr normalization of the linear and quadratic rates (a local fit, not a
# whole-record intercept).
BASELINE_WINDOW_HOURS = 10

## Per-run tables + metadata

Pulls `t_hours` (= TOS) and `R` from the already-filtered `tos_data`, plus per-run metadata (T, P,
flow rate) for labels. Flow is each run's own median in-band H2 + CO2.

In [ ]:
CLEAN_RUNS = {}
RUN_METADATA = {}

for name, cfg in FILTER_CONFIG.items():
    filt = tos_data[name]['filtered']
    CLEAN_RUNS[name] = pd.DataFrame({
        't_hours': filt[TOS_COL].to_numpy(),
        'R': filt[RESISTANCE_COL].to_numpy(),
    }).sort_values('t_hours').reset_index(drop=True)

    T_C = cfg['target_temp']
    RUN_METADATA[name] = {
        'T_C': T_C,
        'T_K': T_C + 273.15,
        'P_bar_mean_measured': filt[PRESSURE_COL].mean(),
    }

# Flow rate: each run's own H2 + CO2 (SLPM), carry-forward matched to its in-band resistance timestamps,
# median over the in-band period. A run whose export lacks H2/CO2 (or has no in-band readings) gets NaN
# and is flagged -- re-export with those traces rather than borrowing another run's value.
def in_band_median(df, col, ref_index):
    """Median of `col`, carried forward (merge_asof backward) onto the in-band timestamps."""
    if col not in df.columns:
        return np.nan
    s = df[col].dropna().rename(col).reset_index().sort_values('timestamp')
    ref = ref_index.to_series().reset_index(drop=True).rename('timestamp').to_frame().sort_values('timestamp')
    return pd.merge_asof(ref, s, on='timestamp', direction='backward')[col].median()


for name in RUN_METADATA:
    df, in_band_index = data[name]['df'], tos_data[name]['filtered'].index
    h2, co2 = in_band_median(df, 'H2', in_band_index), in_band_median(df, 'CO2', in_band_index)
    RUN_METADATA[name].update(H2_SLPM=h2, CO2_SLPM=co2, flow_rate_SLPM=h2 + co2,
                              flow_rate_measured=bool(np.isfinite(h2 + co2)))

for name, md in RUN_METADATA.items():
    flow_txt = (f"flow={md['flow_rate_SLPM']:.2f} SLPM (H2 {md['H2_SLPM']:.2f} + CO2 {md['CO2_SLPM']:.2f}, "
                f"H2:CO2 = {md['H2_SLPM'] / md['CO2_SLPM']:.2f})" if md['flow_rate_measured']
                else 'flow=n/a  [H2/CO2 not in this export -- re-export with those traces]')
    print(f"{name}: T={md['T_C']}C ({md['T_K']:.2f} K), P(mean, in-band)={md['P_bar_mean_measured']:.2f} bar, "
          f"{flow_txt}")


## Fit data: hourly bins, burn-in removed, excursions masked, common window applied

In [ ]:
def bin_hourly(t, R, bin_hours=BIN_HOURS):
    """Reduce a densely logged (t, R) series to hourly bin means."""
    bins = np.floor(t / bin_hours).astype(int)
    g = pd.DataFrame({'bin': bins, 't': t, 'R': R}).groupby('bin').agg(t=('t', 'mean'), R=('R', 'mean'))
    return g['t'].to_numpy(), g['R'].to_numpy()


def detect_excursion(t, R, window_hours=EXCURSION_WINDOW_HOURS, step_hours=EXCURSION_STEP_HOURS,
                     mult=EXCURSION_SLOPE_MULT):
    """Return (start, end) hours of the stretch of rolling windows whose |dR/dt| exceeds `mult` x the
    run's median |dR/dt|, or None if no window is flagged."""
    centers = np.arange(t.min() + window_hours / 2, t.max() - window_hours / 2 + 1e-9, step_hours)
    slopes = []
    for c in centers:
        m = (t >= c - window_hours / 2) & (t < c + window_hours / 2)
        slopes.append(np.polyfit(t[m], R[m], 1)[0] if m.sum() >= 3 else np.nan)
    slopes = np.abs(np.array(slopes))
    flagged = centers[slopes > mult * np.nanmedian(slopes)]
    if len(flagged) == 0:
        return None
    return flagged.min() - step_hours / 2, flagged.max() + step_hours / 2


FIT_DATA = {}
for name in FILTER_CONFIG:
    t, R = bin_hourly(CLEAN_RUNS[name]['t_hours'].to_numpy(), CLEAN_RUNS[name]['R'].to_numpy())
    keep = t >= BURN_IN_HOURS
    excursion = detect_excursion(t[keep], R[keep]) if EXCLUDE_EXCURSIONS else None
    if excursion is not None:
        keep &= t < excursion[0]
    FIT_DATA[name] = {'t_all': t, 'R_all': R, 'keep': keep, 'excursion': excursion}

if FIT_END_HOURS == 'common':
    fit_end = min(d['t_all'][d['keep']].max() for d in FIT_DATA.values())
else:
    fit_end = FIT_END_HOURS  # None or a number

for name, d in FIT_DATA.items():
    keep, t, excursion = d.pop('keep'), d['t_all'], d['excursion']
    usable_end = t[keep].max()
    if fit_end is not None:
        keep &= t <= fit_end
    d['t'], d['R'] = t[keep], d['R_all'][keep]
    d['fit_end'] = t[keep].max()

    msg = (f"excursion [{excursion[0]:.1f}, {excursion[1]:.1f}] h -- data from {excursion[0]:.1f} h on dropped"
           if excursion is not None else 'no excursion flagged')
    trunc = (f"; truncated at {fit_end:.1f} h (usable data to {usable_end:.1f} h)"
             if fit_end is not None and fit_end < usable_end else '')
    print(f"{name}: {keep.sum()} hourly points in [{d['t'].min():.1f}, {d['t'].max():.1f}] h; {msg}{trunc}")

FIT_WINDOW_LABEL = (f"t = {BURN_IN_HOURS:g}-{fit_end:.0f} h" if fit_end is not None
                    else f"t > {BURN_IN_HOURS:g} h, full record per run")
print(f"\nFit window: {FIT_WINDOW_LABEL}")

## Fits with moving-block bootstrap CIs

Linear and quadratic rates are also expressed as %/hr, normalized by the measured resistance at
`t = burn-in` (local linear fit over +/- `BASELINE_WINDOW_HOURS`, bootstrapped the same way). The
exponential `b` is already a fractional rate, so it needs no separate baseline. Lower AIC = better fit
(all models are compared on the same points; the quadratic pays a penalty for its extra parameter).

In [ ]:
def linear_model(x, R0, k):
    return R0 + k * x


def exponential_model(x, R0, b):
    return R0 * np.exp(b * x)


def quadratic_model(x, R0, k, c):
    return R0 + k * x + c * x ** 2


def power_model(x, R0, A, n):
    return R0 + A * np.power(x, n)


def linear_power_model(x, R0, k, A, n):
    return R0 + k * x + A * np.power(x, n)


def geometric_model(x, R0, t_run, n):
    return R0 / (1 - np.power(x / t_run, n))


def fit_linear(x, R):
    res = stats.linregress(x, R)
    return np.array([res.intercept, res.slope])  # (R0, k)


def fit_exponential(x, R):
    b0, lnR0 = np.polyfit(x, np.log(R), 1)  # log-linear starting guess
    popt, _ = curve_fit(exponential_model, x, R, p0=[np.exp(lnR0), b0], maxfev=20000)
    return popt  # (R0, b)


def fit_quadratic(x, R):
    c, k, R0 = np.polyfit(x, R, 2)
    return np.array([R0, k, c])


def fit_power(x, R):
    n0 = float(np.clip(1.5, *POWER_N_BOUNDS))
    A0, R00 = np.polyfit(np.power(x, n0), R, 1)  # linear in (R0, A) at a fixed starting n
    popt, _ = curve_fit(power_model, x, R, p0=[R00, A0, n0],
                        bounds=([-np.inf, -np.inf, POWER_N_BOUNDS[0]], [np.inf, np.inf, POWER_N_BOUNDS[1]]),
                        maxfev=20000)
    return popt  # (R0, A, n)


def fit_linear_power(x, R):
    n0 = float(np.clip(2.0, *POWER_N_BOUNDS))
    A0, k0, R00 = np.linalg.lstsq(np.column_stack([np.power(x, n0), x, np.ones_like(x)]), R, rcond=None)[0]
    popt, _ = curve_fit(linear_power_model, x, R, p0=[R00, k0, A0, n0],
                        bounds=([-np.inf, -np.inf, -np.inf, POWER_N_BOUNDS[0]],
                                [np.inf, np.inf, np.inf, POWER_N_BOUNDS[1]]),
                        maxfev=20000)
    return popt  # (R0, k, A, n)


def fit_geometric(x, R):
    T = x.max()
    n0 = float(np.clip(0.5, *GEOM_N_BOUNDS))
    f_end = np.clip(1 - R[0] / R[-1], 0.05, 0.95)  # fraction of conducting section lost by the end
    t0 = T / f_end ** (1 / n0)
    popt, _ = curve_fit(geometric_model, x, R, p0=[R[0], t0, n0],
                        bounds=([0, T * 1.001, GEOM_N_BOUNDS[0]],
                                [np.inf, T * GEOM_TRUN_MAX_MULT, GEOM_N_BOUNDS[1]]),
                        maxfev=20000)
    return popt  # (R0, t_run, n)


# 'rate': absolute rate in Ohm/hr from params `th` (last axis) over a window of length T (None if n/a).
# 'pct': %/hr; defaults to 100 * rate / measured baseline R when not given.
MODELS = {
    'linear': {'fn': linear_model, 'fit': fit_linear, 'n_params': 2, 'color': 'C0', 'ls': '-',
               'rate': lambda th, T: th[..., 1]},
    'exponential': {'fn': exponential_model, 'fit': fit_exponential, 'n_params': 2, 'color': 'C3', 'ls': '--',
                    'rate': None, 'pct': lambda th, T: 100 * np.expm1(th[..., 1])},
    'quadratic': {'fn': quadratic_model, 'fit': fit_quadratic, 'n_params': 3, 'color': 'C2', 'ls': '-.',
                  'rate': lambda th, T: th[..., 1] + th[..., 2] * T},  # average slope over [0, T]
    'power': {'fn': power_model, 'fit': fit_power, 'n_params': 3, 'color': 'C4', 'ls': ':',
              'rate': lambda th, T: th[..., 1] * np.power(T, th[..., 2] - 1)},  # average slope over [0, T]
    'linear_power': {'fn': linear_power_model, 'fit': fit_linear_power, 'n_params': 4, 'color': 'C1',
                     'ls': (0, (5, 1, 1, 1)),
                     'rate': lambda th, T: th[..., 1] + th[..., 2] * np.power(T, th[..., 3] - 1)},  # avg slope
    'geometric': {'fn': geometric_model, 'fit': fit_geometric, 'n_params': 3, 'color': 'C5', 'ls': '-',
                  'rate': lambda th, T: (geometric_model(T, th[..., 0], th[..., 1], th[..., 2]) - th[..., 0]) / T},
}


def moving_block_bootstrap(x, R, fit_fn, model_fn, n_boot=N_BOOT, block_hours=BOOT_BLOCK_HOURS, rng=RNG):
    """Resample contiguous blocks of residuals (with replacement) onto the base fit, refit each time.
    Block length is capped at a quarter of the record span so short records still get real resampling.
    Returns (base_params, bootstrap_param_draws)."""
    theta = fit_fn(x, R)
    pred = model_fn(x, *theta)
    resid = R - pred
    n = len(x)
    dt = np.median(np.diff(x))
    block_len = max(2, min(n, int(round(min(block_hours, (x.max() - x.min()) / 4) / dt))))
    n_blocks = int(np.ceil(n / block_len))

    draws = []
    for _ in range(n_boot):
        starts = rng.integers(0, n - block_len + 1, size=n_blocks)
        idx = np.concatenate([np.arange(s, s + block_len) for s in starts])[:n]
        try:
            draws.append(fit_fn(x, pred + resid[idx]))
        except RuntimeError:  # curve_fit non-convergence on a rare draw
            continue
    return theta, np.array(draws)


def aic(R, pred, n_params):
    n = len(R)
    return n * np.log(np.sum((R - pred) ** 2) / n) + 2 * n_params


def ci95(draws):
    return tuple(np.percentile(draws, [2.5, 97.5]))


FITS = {}
for name, d in FIT_DATA.items():
    x, R = d['t'] - BURN_IN_HOURS, d['R']
    T = x.max()

    # Measured baseline at t = burn-in (local fit, bootstrapped) for the %/hr normalization.
    bm = np.abs(d['t_all'] - BURN_IN_HOURS) <= BASELINE_WINDOW_HOURS
    base, base_boot = moving_block_bootstrap(d['t_all'][bm] - BURN_IN_HOURS, d['R_all'][bm],
                                             fit_linear, linear_model)
    R_base, R_base_draws = base[0], base_boot[:, 0]

    FITS[name] = {'x': x, 'R': R, 'T': T, 'R_base': R_base, 'models': {}}
    print(f"{name}  ({len(x)} hourly points, t = {BURN_IN_HOURS}-{d['t'].max():.0f} h, "
          f"baseline R = {R_base:.5f} Ohm)")

    for m in FIT_MODELS:
        spec = MODELS[m]
        theta, boot = moving_block_bootstrap(x, R, spec['fit'], spec['fn'])
        res = {'theta': theta, 'boot': boot, 'n_boot': len(boot),
               'aic': aic(R, spec['fn'](x, *theta), spec['n_params'])}

        if spec['rate'] is not None:
            res['rate'] = spec['rate'](theta, T)
            res['rate_ci'] = ci95(spec['rate'](boot, T))
        if 'pct' in spec:
            res['pct'] = spec['pct'](theta, T)
            res['pct_ci'] = ci95(spec['pct'](boot, T))
        else:
            n_pair = min(len(boot), len(R_base_draws))
            res['pct'] = 100 * res['rate'] / R_base
            res['pct_ci'] = ci95(100 * spec['rate'](boot[:n_pair], T) / R_base_draws[:n_pair])
        if m == 'quadratic':
            res['c_ci'] = ci95(boot[:, 2])
        if m == 'power':
            res['n_ci'] = ci95(boot[:, 2])
        if m == 'linear_power':
            res['n_ci'] = ci95(boot[:, 3])
        if m == 'geometric':
            res['n_ci'] = ci95(boot[:, 2])
            res['t_run_ci'] = ci95(boot[:, 1])
        FITS[name]['models'][m] = res

        line = f"  {m:<12} AIC = {res['aic']:8.1f}   "
        if 'rate' in res:
            line += (f"rate = {res['rate']*1e3:.4f} mOhm/hr [{res['rate_ci'][0]*1e3:.4f}, "
                     f"{res['rate_ci'][1]*1e3:.4f}]   ")
        line += f"= {res['pct']:.4f} %/hr [{res['pct_ci'][0]:.4f}, {res['pct_ci'][1]:.4f}]"
        if res['n_boot'] < N_BOOT:
            line += f"   ({res['n_boot']}/{N_BOOT} refits converged)"
        print(line)
        if m == 'quadratic':
            R0, k, c = theta
            print(f"  {'':<12} slope {k*1e3:.4f} -> {(k + 2*c*T)*1e3:.4f} mOhm/hr over the window; "
                  f"c = {c*1e6:.4f} uOhm/hr^2 [{res['c_ci'][0]*1e6:.4f}, {res['c_ci'][1]*1e6:.4f}]")
        if m == 'power':
            R0, A, n = theta
            at_bound = ('  <-- n at its bound; data wants n outside POWER_N_BOUNDS'
                        if np.isclose(n, POWER_N_BOUNDS, rtol=0, atol=1e-3).any() else '')
            print(f"  {'':<12} n = {n:.3f} [{res['n_ci'][0]:.3f}, {res['n_ci'][1]:.3f}]; slope "
                  f"{A*n*x[0]**(n-1)*1e3:.4f} (t={x[0]+BURN_IN_HOURS:.0f}h) -> {A*n*T**(n-1)*1e3:.4f} mOhm/hr"
                  f"{at_bound}")
        if m == 'linear_power':
            R0, k, A, n = theta
            at_bound = ('  <-- n at its bound; data wants n outside POWER_N_BOUNDS'
                        if np.isclose(n, POWER_N_BOUNDS, rtol=0, atol=1e-3).any() else '')
            print(f"  {'':<12} n = {n:.3f} [{res['n_ci'][0]:.3f}, {res['n_ci'][1]:.3f}]; slope "
                  f"{(k + A*n*x[0]**(n-1))*1e3:.4f} (t={x[0]+BURN_IN_HOURS:.0f}h) -> "
                  f"{(k + A*n*T**(n-1))*1e3:.4f} mOhm/hr{at_bound}")
        if m == 'geometric':
            R0, t_run, n = theta
            slope = lambda xx: R0 * n * xx ** (n - 1) / t_run ** n / (1 - (xx / t_run) ** n) ** 2
            flags = []
            if np.isclose(n, GEOM_N_BOUNDS, rtol=0, atol=1e-3).any():
                flags.append('n at its bound')
            if np.isclose(t_run, [T * 1.001, T * GEOM_TRUN_MAX_MULT], rtol=1e-3).any():
                flags.append('t_run at its bound')
            print(f"  {'':<12} n = {n:.3f} [{res['n_ci'][0]:.3f}, {res['n_ci'][1]:.3f}]; slope "
                  f"{slope(x[0])*1e3:.4f} (t={x[0]+BURN_IN_HOURS:.0f}h) -> {slope(T)*1e3:.4f} mOhm/hr"
                  + (f"  <-- {', '.join(flags)}" if flags else ''))
            print(f"  {'':<12} runaway time (extrapolation, R -> inf) at TOS = {t_run + BURN_IN_HOURS:.0f} h "
                  f"[{res['t_run_ci'][0] + BURN_IN_HOURS:.0f}, {res['t_run_ci'][1] + BURN_IN_HOURS:.0f}]"
                  f"  ({t_run / T:.1f}x the fit window)")

    best = min(FITS[name]['models'], key=lambda m: FITS[name]['models'][m]['aic'])
    print(f"  -> best by AIC: {best}")
    print()

### Fits vs. data (top) and residuals (bottom)

Gray = all hourly-binned points; colored = points used in the fit. Shaded = burn-in and anything past
the fit window (masked excursion and/or common-window truncation). Residuals that wander systematically (rather than scatter around 0) mean the model shape
doesn't capture the trend -- that autocorrelation is why the CIs come from a block bootstrap.

In [ ]:
fig, axes = plt.subplots(2, len(FITS), figsize=(7 * len(FITS), 8), sharex='col',
                         gridspec_kw={'height_ratios': [3, 1.3]}, squeeze=False)

for col, (name, f) in enumerate(FITS.items()):
    d = FIT_DATA[name]
    ax, ax_r = axes[0, col], axes[1, col]
    x_line = np.linspace(0, f['T'], 300)

    ax.plot(d['t_all'], d['R_all'], '.', color='0.75', markersize=3, label='excluded')
    ax.plot(d['t'], d['R'], '.', color='k', markersize=3, label='fit data')
    for m, res in f['models'].items():
        spec = MODELS[m]
        ax.plot(x_line + BURN_IN_HOURS, spec['fn'](x_line, *res['theta']), color=spec['color'], lw=2,
                ls=spec['ls'], label=m)
        ax_r.plot(d['t'], f['R'] - spec['fn'](f['x'], *res['theta']), '.', color=spec['color'],
                  markersize=3, label=m)
    for a_ in (ax, ax_r):
        a_.axvspan(0, BURN_IN_HOURS, color='0.9', zorder=0)
        if d['t_all'].max() > d['fit_end']:
            a_.axvspan(d['fit_end'], d['t_all'].max(), color='0.9', zorder=0)
    ax.set_title(f"{name} ({RUN_METADATA[name]['T_C']}°C)")
    ax.set_ylabel(f'{RESISTANCE_COL} (Ohm)')
    ax.legend(fontsize=9)
    ax.grid(alpha=0.3)

    ax_r.axhline(0, color='k', lw=0.8)
    ax_r.set_xlabel(TOS_COL)
    ax_r.set_ylabel('residual (Ohm)')
    ax_r.legend(fontsize=9)
    ax_r.grid(alpha=0.3)

fig.tight_layout()
plt.show()

## 2W Resistance Increase Rate by Run

Saved to `OUTPUT_DIR/2W_resistance_increase_rate.png` (Ohm/hr) and `OUTPUT_DIR/2W_resistance_increase_rate_pct.png` (%/hr).

In [ ]:
MODEL_COLORS = {'linear': '#2a78d6', 'exponential': '#c8453a', 'quadratic': '#2e9d5b', 'power': '#7b52ab',
                'linear_power': '#d9822b', 'geometric': '#d55181'}
MODEL_LABELS = {'linear': 'linear', 'exponential': 'exponential',
                'quadratic': 'quadratic (avg. slope over window)',
                'power': 'power law (avg. slope over window)',
                'linear_power': 'linear + power (avg. slope over window)',
                'geometric': 'geometric (avg. slope over window)'}
names = list(FITS.keys())
labels = [f"{n.replace('_', '-')}, {RUN_METADATA[n]['T_C']}°C" for n in names]
x = np.arange(len(names))

plt.rcParams.update({
    'font.size': 13, 'axes.spines.top': False, 'axes.spines.right': False,
    'axes.edgecolor': '#8a8a86', 'axes.labelcolor': '#0b0b0b', 'text.color': '#0b0b0b',
    'xtick.color': '#52514e', 'ytick.color': '#52514e',
})


def rate_errorbar(ax, xs, vals, ci, color, label, fmt, unit):
    vals = np.asarray(vals)
    lo, hi = np.array([c[0] for c in ci]), np.array([c[1] for c in ci])
    ax.errorbar(xs, vals, yerr=[vals - lo, hi - vals], fmt='o', markersize=10, capsize=7, color=color,
                elinewidth=2, capthick=2, zorder=5, label=label)
    for xi, v, l, h in zip(xs, vals, lo, hi):
        ax.annotate(f'{v:{fmt}} {unit}\n[{l:{fmt}}, {h:{fmt}}]', (xi, h), textcoords='offset points',
                    xytext=(0, 8), ha='center', va='bottom', fontsize=9, color=color)
    return hi.max()


def rate_plot(key, ci_key, models, scale, unit, ylabel, title, path):
    width = min(0.26, 0.8 / len(models)) if len(models) > 1 else 0
    offsets = (np.arange(len(models)) - (len(models) - 1) / 2) * width
    fig, ax = plt.subplots(figsize=(4 + 2.2 * len(models), 5.5), dpi=150)
    top = 0
    for off, m in zip(offsets, models):
        vals = [FITS[n]['models'][m][key] * scale for n in names]
        cis = [tuple(c * scale for c in FITS[n]['models'][m][ci_key]) for n in names]
        top = max(top, rate_errorbar(ax, x + off, vals, cis, MODEL_COLORS[m], MODEL_LABELS[m], '.3f', unit))
    ax.set_xticks(x)
    ax.set_xticklabels(labels, fontsize=13)
    ax.set_xlim(-0.6, len(labels) - 0.4)
    ax.set_ylim(0, top * 1.4)
    ax.set_ylabel(ylabel)
    ax.set_title(title, fontsize=15, pad=14)
    ax.grid(axis='y', alpha=0.25, color='#c3c2b7')
    ax.set_axisbelow(True)
    ax.legend(loc='upper left', fontsize=10, frameon=False)
    fig.tight_layout()
    fig.savefig(OUTPUT_DIR / path, dpi=300, bbox_inches='tight', facecolor='white')
    plt.show()


# Absolute rate (Ohm/hr) -- models with an absolute rate (exponential's is relative only)
abs_models = [m for m in FIT_MODELS if MODELS[m]['rate'] is not None]
if abs_models:
    rate_plot('rate', 'rate_ci', abs_models, 1e3, 'mΩ/hr', '2W resistance increase rate (mΩ/hr)',
              f'2W Resistance Increase Rate by Run\n({FIT_WINDOW_LABEL})', '2W_resistance_increase_rate.png')

# Relative rate (%/hr) -- all fitted models
rate_plot('pct', 'pct_ci', FIT_MODELS, 1, '%/hr', '2W resistance increase rate (% / hr)',
          f'2W Resistance Increase Rate by Run\n(% per hour, {FIT_WINDOW_LABEL})',
          '2W_resistance_increase_rate_pct.png')

## Summary

In [ ]:
def fmt_ci(v, ci, scale=1.0):
    return f"{v*scale:.4f} [{ci[0]*scale:.4f}, {ci[1]*scale:.4f}]"


rows = {}
for name, f in FITS.items():
    col = {
        'T (C)': RUN_METADATA[name]['T_C'],
        'fit window (h)': f"{FIT_DATA[name]['t'].min():.0f}-{FIT_DATA[name]['t'].max():.0f}",
        'n points': len(f['x']),
    }
    for m, res in f['models'].items():
        if 'rate' in res:
            col[f'{m} rate (mOhm/hr)'] = fmt_ci(res['rate'], res['rate_ci'], 1e3)
        col[f'{m} (%/hr)'] = fmt_ci(res['pct'], res['pct_ci'])
        col[f'{m} AIC'] = round(res['aic'], 1)
        if m in ('power', 'linear_power', 'geometric'):
            col[f'{m} exponent n'] = f"{res['theta'][-1]:.3f} [{res['n_ci'][0]:.3f}, {res['n_ci'][1]:.3f}]"
        if m == 'geometric':
            t_run, ci = res['theta'][1] + BURN_IN_HOURS, [c + BURN_IN_HOURS for c in res['t_run_ci']]
            col['geometric runaway TOS (h, extrapolated)'] = f"{t_run:.0f} [{ci[0]:.0f}, {ci[1]:.0f}]"
    rows[name] = col

summary = pd.DataFrame(rows)
summary

## Fitted parameters with 95% CIs

Every parameter of every fitted model, per run, with its moving-block bootstrap 95% CI (2.5 / 97.5
percentiles of the refits). `t` in every model is hours since the end of burn-in, so `R0` is the fitted
resistance at TOS = `BURN_IN_HOURS`, and the geometric `t_run` is likewise counted from burn-in (add
`BURN_IN_HOURS` for TOS). Units of the power-law `A` depend on its fitted `n`. A parameter fitted at one
of its bounds (`POWER_N_BOUNDS`, `GEOM_N_BOUNDS`, the `t_run` limits) has a CI clipped at that bound --
the printout from the fit cell flags those.

Saved in long format to `OUTPUT_DIR/model_parameters.csv`.

In [ ]:
PARAM_INFO = {  # model -> [(parameter, units, display scale)], in the model's parameter order
    'linear':       [('R0', 'Ohm', 1), ('k', 'mOhm/hr', 1e3)],
    'exponential':  [('R0', 'Ohm', 1), ('b', '1/hr', 1)],
    'quadratic':    [('R0', 'Ohm', 1), ('k', 'mOhm/hr', 1e3), ('c', 'uOhm/hr^2', 1e6)],
    'power':        [('R0', 'Ohm', 1), ('A', 'Ohm/hr^n', 1), ('n', '-', 1)],
    'linear_power': [('R0', 'Ohm', 1), ('k', 'mOhm/hr', 1e3), ('A', 'Ohm/hr^n', 1), ('n', '-', 1)],
    'geometric':    [('R0', 'Ohm', 1), ('t_run', 'hr after burn-in', 1), ('n', '-', 1)],
}

rows = []
for name, f in FITS.items():
    for m, res in f['models'].items():
        for j, (param, units, scale) in enumerate(PARAM_INFO[m]):
            lo, hi = ci95(res['boot'][:, j])
            rows.append({'run': name, 'model': m, 'parameter': param, 'units': units,
                         'estimate': res['theta'][j] * scale, 'ci_low': lo * scale, 'ci_high': hi * scale,
                         'n_boot': res['n_boot']})

PARAM_TABLE = pd.DataFrame(rows)
try:
    PARAM_TABLE.to_csv(OUTPUT_DIR / 'model_parameters.csv', index=False)
except PermissionError:  # usually the CSV is open in Excel, which locks it
    print('WARNING: model_parameters.csv is locked (open in Excel?) -- close it and re-run this cell '
          'to update it. Table below is current.')

# Display view: one row per (model, parameter), one column per run.
row_order = pd.MultiIndex.from_tuples(
    list(dict.fromkeys(zip(PARAM_TABLE['model'], PARAM_TABLE['parameter'], PARAM_TABLE['units']))),
    names=['model', 'parameter', 'units'])  # keep each model's own parameter order (unstack sorts)
param_view = (PARAM_TABLE
              .assign(value=[f"{e:.4g} [{l:.4g}, {h:.4g}]"
                             for e, l, h in zip(PARAM_TABLE['estimate'], PARAM_TABLE['ci_low'], PARAM_TABLE['ci_high'])])
              .set_index(['model', 'parameter', 'units', 'run'])['value']
              .unstack('run')
              .reindex(row_order))
param_view.columns.name = None
print(f"Fit window: {FIT_WINDOW_LABEL};  estimate [95% CI]")
param_view